# Exercice 2 — Data augmentation par décalage

**Hands-On ML, chapitre 3, exercice 2** · carnet CS229, bloc **B14**, étape 13 (option)

> *Écrivez une fonction qui décale une image MNIST d'un pixel dans n'importe quelle direction.
> Puis, pour chaque image du training set, créez quatre copies décalées (une par direction) et
> ajoutez-les au training set. Enfin, entraînez votre meilleur modèle sur ce training set étendu
> et mesurez son accuracy sur le test set. Vous devriez constater que le modèle s'est amélioré.
> Cette technique d'enrichissement artificiel du training set s'appelle la **data augmentation**
> ou l'**expansion du training set**.*

---

## L'idée en une phrase

Le modèle ne sait pas qu'**un 7 décalé d'un pixel reste un 7**. C'est vous qui le savez. La data
augmentation est le canal par lequel vous lui transmettez cette connaissance.

## Pourquoi ça marche, formellement

Vous ne créez pas d'information nouvelle — les copies décalées sont déterministes. Vous imposez
une **invariance** : vous forcez le modèle à produire la même sortie sur toute une orbite de
transformations.

$$h_{\theta}\left(T(x)\right) = h_{\theta}(x) \quad \text{pour toute translation } T
\text{ d'un pixel}$$

C'est une forme de **régularisation** (bloc **B13**) : vous ne touchez pas au modèle, vous élargissez
la distribution d'entraînement, ce qui réduit la variance. Au sens de la décomposition du point
**B12-3** :

$$\text{MSE}(x) = \underbrace{\sigma^{2}}_{\text{bruit}} +
\underbrace{\left(h^{*}(x) - h_{\text{avg}}(x)\right)^{2}}_{\text{bias}^{2}} +
\underbrace{\operatorname{var}\left(h_{S}(x)\right)}_{\text{variance}}$$

on attaque le troisième terme.

## Durée

Comptez **1 h 30**, dont une bonne part d'attente.

## Prérequis

Avoir fait l'exercice 1. Ce notebook y fait référence comme baseline.

---
## 1. Environnement

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from scipy.ndimage import shift

from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix

SEED = 42
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 110

print("numpy", np.__version__)

In [ ]:
mnist = fetch_openml("mnist_784", as_frame=False, parser="auto")

X = mnist.data.astype(np.uint8)
y = mnist.target.astype(np.uint8)

X_train_full, X_test = X[:60000], X[60000:]
y_train_full, y_test = y[:60000], y[60000:]

print("train complet :", X_train_full.shape)
print("test          :", X_test.shape)

---
## 2. Le protocole de comparaison — à lire avant de coder

Cet exercice demande de **comparer deux modèles** : avec et sans augmentation. Deux évaluations.

Or le test set ne se touche **qu'une seule fois** (point **B13-6**). Si on évalue les deux
variantes dessus et qu'on garde la meilleure, on a choisi un hyperparamètre sur le test set : le
score annoncé devient optimiste.

**La solution** : découper le training set en deux.

| Jeu | Taille | Rôle |
|---|---|---|
| `X_train` | 50 000 | Entraînement, avec ou sans augmentation |
| `X_val` | 10 000 | Comparaison des deux variantes |
| `X_test` | 10 000 | Évaluation finale, une seule fois, du gagnant |

C'est la **hold-out cross validation** du point **B13-6**. Elle est plus légère qu'une k-fold ici,
parce qu'un entraînement sur 250 000 images augmentées n'est pas quelque chose qu'on répète cinq
fois pour le plaisir.

In [ ]:
X_train, X_val = X_train_full[:50000], X_train_full[50000:]
y_train, y_val = y_train_full[:50000], y_train_full[50000:]

print("train      :", X_train.shape)
print("validation :", X_val.shape)
print("test       :", X_test.shape, " <- on n'y touche qu'a la section 9")

**Règle absolue** : on augmente **uniquement** `X_train`.

Augmenter la validation ou le test n'aurait aucun sens : on mesurerait la performance sur des
images artificielles, pas sur des chiffres réels. Et augmenter **avant** le split serait un
**data leakage** caractérisé (point **B1M-3**) — la copie décalée d'une image d'entraînement se
retrouverait dans la validation, qui deviendrait presque un second training set.

---
## 3. La fonction de décalage

`scipy.ndimage.shift` décale un tableau. Trois points d'attention :

- l'image arrive **aplatie** (784 valeurs), il faut la remettre en 28 × 28 avant, et l'aplatir
  après ;
- l'ordre des axes est `[dy, dx]` : **ligne d'abord, colonne ensuite** ;
- `cval=0` remplit avec du noir les pixels qui entrent dans le cadre, `mode="constant"` dit
  d'utiliser cette valeur plutôt que de répéter le bord.

In [ ]:
def shift_image(image, dx, dy):
    # image : vecteur aplati de 784 pixels. dx > 0 decale vers la droite,
    # dy > 0 decale vers le bas. Retourne un vecteur aplati.
    image = image.reshape((28, 28))
    shifted = shift(image, [dy, dx], cval=0, mode="constant")
    return shifted.reshape([-1])

### Vérification visuelle

On ne fait jamais confiance à une transformation de données sans la regarder.

In [ ]:
img = X_train[1000]
directions = [(0, 0, "original"), (1, 0, "droite"), (-1, 0, "gauche"),
              (0, 1, "bas"), (0, -1, "haut")]

fig, axes = plt.subplots(1, 5, figsize=(10, 2.4))
for ax, (dx, dy, nom) in zip(axes, directions):
    ax.imshow(shift_image(img, dx, dy).reshape(28, 28), cmap="binary")
    ax.set_title(nom, fontsize=10)
    ax.axis("off")
fig.suptitle(f"Chiffre {y_train[1000]} et ses 4 decalages", y=1.08)
plt.tight_layout()
plt.show()

### Un détail qui mérite d'être vérifié plutôt que supposé

Par défaut, `shift` interpole par **spline cubique** (`order=3`). Sur une image en virgule
flottante, cette interpolation peut produire de légères valeurs **négatives** ou **supérieures à
255** au voisinage d'un trait net : c'est le phénomène de dépassement classique des splines.

Mais `scipy.ndimage.shift` **conserve le type du tableau d'entrée**. Comme notre `X` est en
`uint8`, scipy écrête lui-même et rend du `uint8` propre. Vérifions-le au lieu de le croire.

In [ ]:
for nom, entree in [("uint8  ", X_train[1000]),
                    ("float64", X_train[1000].astype(np.float64))]:
    for dx in [1, 0.5]:
        s = shift_image(entree, dx, 0)
        print(f"entree {nom}  dx={dx:<4} -> dtype {str(s.dtype):8s} "
              f"min {float(s.min()):8.2f}  max {float(s.max()):8.2f}")

Deux faits, qu'aucune intuition ne donnait :

**À décalage entier, la spline est exacte.** Il n'y a aucune interpolation : les coefficients
valent exactement 0 ou 1, le pixel est recopié tel quel. Aucun dépassement, quel que soit le type
d'entrée. Le dépassement n'apparaît qu'à **décalage fractionnaire** (`dx=0.5`), où il est massif
— de l'ordre de $-43$ à $+289$ sur cette image.

**`scipy.ndimage.shift` conserve le type d'entrée.** Avec du `uint8`, scipy écrête lui-même et
rend du `uint8` propre, même en fractionnaire.

**Conséquence.** Le `np.clip(..., 0, 255)` de la section suivante est ici un **no-op strict** :
décalage entier et entrée `uint8`, il n'y a rien à écrêter.

On le garde quand même, et ce n'est pas de la superstition. Dès qu'on étend l'augmentation à des
**rotations** ou à des décalages **sous-pixel** — ce que fait toute vraie pipeline d'augmentation
— le dépassement devient réel. Et le cast en `uint8` ne sature pas, il **boucle** : un 288,6
devient 32, un $-42{,}8$ devient 213. Silencieusement.

Le `.astype(np.uint8)`, lui, n'est pas optionnel : il divise la mémoire par huit.

In [ ]:
print("sans clip, un 288.6 caste en uint8 donne", np.array([288.6]).astype(np.uint8)[0])
print("sans clip, un -42.8 caste en uint8 donne", np.array([-42.8]).astype(np.uint8)[0])

---
## 4. Construction du training set augmenté

### D'abord, le calcul de mémoire

C'est là que beaucoup de notebooks partent en `MemoryError`. Le training set passe de 50 000 à
250 000 images.

$$250\,000 \times 784 \times 8 \text{ octets} \approx 1{,}6 \text{ Go en float64}$$
$$250\,000 \times 784 \times 1 \text{ octet} \approx 196 \text{ Mo en uint8}$$

Le facteur huit n'est pas une optimisation cosmétique : il décide si la cellule tourne ou si votre
machine se met à swapper.

In [ ]:
n_aug = 5 * len(X_train)
for dtype, octets in [("float64", 8), ("float32", 4), ("uint8", 1)]:
    print(f"{dtype:8s} : {n_aug * 784 * octets / 1e9:5.2f} Go")

### Construction

On préalloue le tableau de sortie plutôt que d'empiler des listes : c'est plus rapide et le pic
mémoire est maîtrisé.

**Cellule longue** : comptez une à trois minutes pour 200 000 appels à `shift`.

In [ ]:
DECALAGES = [(1, 0), (-1, 0), (0, 1), (0, -1)]

t0 = time.perf_counter()

n_orig = len(X_train)
X_train_aug = np.empty((n_orig * 5, 784), dtype=np.uint8)
y_train_aug = np.empty(n_orig * 5, dtype=np.uint8)

# Bloc 0 : les images originales
X_train_aug[:n_orig] = X_train
y_train_aug[:n_orig] = y_train

# Blocs 1 a 4 : une direction par bloc
for k, (dx, dy) in enumerate(DECALAGES, start=1):
    debut, fin = k * n_orig, (k + 1) * n_orig
    for i, image in enumerate(X_train):
        X_train_aug[debut + i] = np.clip(shift_image(image, dx, dy), 0, 255)
    y_train_aug[debut:fin] = y_train
    print(f"  direction {k}/4 ({dx:+d},{dy:+d}) terminee")

print(f"\nduree : {time.perf_counter() - t0:.1f} s")
print("training set augmente :", X_train_aug.shape)
print(f"memoire : {X_train_aug.nbytes / 1e6:.0f} Mo")

### Le mélange, qui n'est pas optionnel

Tel quel, le tableau est rangé par bloc : 50 000 originales, puis 50 000 décalées à droite, etc.

Cela pose problème pour tout algorithme sensible à l'ordre : la **stochastic gradient descent**
(point **B11-4**) verrait cinquante mille images non décalées d'affilée, puis cinquante mille
décalées à droite. Les gradients successifs seraient fortement corrélés et la convergence en
souffrirait.

Le `RandomForestClassifier` y est insensible — il échantillonne lui-même — mais on mélange quand
même, par principe et pour que ce training set soit réutilisable avec n'importe quel modèle.

In [ ]:
perm = rng.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

print("melange effectue")
print("20 premieres etiquettes :", y_train_aug[:20])

---
## 5. Choix du modèle de comparaison

L'exercice 1 a montré qu'un k-NN à 50 000 exemples mémorisés met déjà une à deux minutes à
prédire 10 000 images. Avec 250 000 exemples mémorisés, comptez **cinq fois plus** : une dizaine
de minutes par évaluation, et autant pour la validation.

On fait donc la comparaison principale avec un `RandomForestClassifier`, dont le coût de
prédiction est indépendant de la taille du training set — c'est un algorithme **paramétrique** au
sens où il résume les données en arbres. Le k-NN sera traité en bonus à la section 8, avec un
avertissement.

Ce choix ne trahit pas l'exercice : la question posée est « la data augmentation améliore-t-elle le
modèle ? », pas « de combien s'améliore le k-NN précisément ».

In [ ]:
def entraine_et_evalue(X_fit, y_fit, nom):
    # Entraine une Random Forest et renvoie son accuracy sur la validation.
    clf = RandomForestClassifier(
        n_estimators=100,
        n_jobs=-1,
        random_state=SEED,
    )
    t0 = time.perf_counter()
    clf.fit(X_fit, y_fit)
    t_fit = time.perf_counter() - t0

    acc = accuracy_score(y_val, clf.predict(X_val))
    print(f"{nom:28s} fit {t_fit:6.1f} s   accuracy val {acc:.4f}")
    return clf, acc

---
## 6. Baseline — sans augmentation

In [ ]:
rf_base, acc_base = entraine_et_evalue(X_train, y_train, "sans augmentation (50k)")

---
## 7. Avec augmentation

**Cellule longue** : cinq fois plus de données, donc un `fit` sensiblement plus lent.

In [ ]:
rf_aug, acc_aug = entraine_et_evalue(X_train_aug, y_train_aug, "avec augmentation (250k)")

In [ ]:
gain = acc_aug - acc_base
erreurs_base = (1 - acc_base) * len(X_val)
erreurs_aug = (1 - acc_aug) * len(X_val)

print(f"accuracy sans augmentation : {acc_base:.4f}")
print(f"accuracy avec augmentation : {acc_aug:.4f}")
print(f"\ngain : {gain:+.4f}  ({gain*100:+.2f} points)")
print(f"erreurs sur la validation : {erreurs_base:.0f} -> {erreurs_aug:.0f}")
print(f"reduction relative du taux d'erreur : {(1 - erreurs_aug/erreurs_base)*100:.1f} %")

### Comment lire ce gain

Le gain absolu paraît modeste — typiquement entre 0,3 et 0,8 point. Lisez-le plutôt en
**réduction relative du taux d'erreur** : passer de 3,0 % à 2,4 % d'erreur, c'est 20 % d'erreurs
en moins, ce qui est considérable à ce niveau de performance.

Et vérifiez que le gain dépasse le bruit de mesure. Avec $n = 10\,000$ en validation, la marge à
95 % est d'environ $\pm 0{,}3$ point (point **B03-13**).

In [ ]:
for nom, a in [("sans augmentation", acc_base), ("avec augmentation", acc_aug)]:
    marge = 1.96 * np.sqrt(a * (1 - a) / len(X_val))
    print(f"{nom:20s} : {a:.4f}  IC 95 % [{a-marge:.4f} ; {a+marge:.4f}]")

print("\nLes intervalles se recouvrent-ils ?",
      "oui -> gain non concluant" if (acc_base + 1.96*np.sqrt(acc_base*(1-acc_base)/len(X_val))
                                      > acc_aug - 1.96*np.sqrt(acc_aug*(1-acc_aug)/len(X_val)))
      else "non -> gain reel")

---
## 8. Bonus : le k-NN augmenté

**Cellule très longue — lisez avant de lancer.**

Avec 250 000 exemples mémorisés, chaque prédiction coûte cinq fois plus qu'à l'exercice 1.
Estimez d'abord, décidez ensuite. On mesure sur un petit échantillon et on extrapole.

In [ ]:
knn_sonde = KNeighborsClassifier(n_neighbors=4, weights="distance", n_jobs=-1)
knn_sonde.fit(X_train_aug[:10_000], y_train_aug[:10_000])

t0 = time.perf_counter()
knn_sonde.predict(X_val[:500])
t_sonde = time.perf_counter() - t0

t_estime = t_sonde * (len(X_train_aug) / 10_000) * (len(X_val) / 500)
print(f"sonde : {t_sonde:.2f} s (10k memorises, 500 requetes)")
print(f"\nestimation pour 250k memorises et 10k requetes : ~{t_estime/60:.0f} min")
print("\nSi ce chiffre vous convient, executez la cellule suivante.")

In [ ]:
# DECOMMENTER POUR LANCER (plusieurs dizaines de minutes)
#
# knn_aug = KNeighborsClassifier(n_neighbors=4, weights="distance", n_jobs=-1)
# knn_aug.fit(X_train_aug, y_train_aug)
#
# t0 = time.perf_counter()
# acc_knn_aug = accuracy_score(y_val, knn_aug.predict(X_val))
# print(f"duree : {(time.perf_counter() - t0)/60:.1f} min")
# print(f"k-NN augmente, accuracy validation : {acc_knn_aug:.4f}")

Pour référence, le k-NN de l'exercice 1 gagne environ **0,5 point** avec l'augmentation, ce qui le
porte autour de 97,6 %. C'est le modèle qui en profite le plus, et la raison est instructive.

Le k-NN compare pixel à pixel. Un 7 décalé d'un pixel est, pour lui, une image **très** différente
— c'est exactement la fragilité repérée à la section 4 de l'exercice 1. En peuplant le voisinage
de chaque image avec ses translatés, vous réparez précisément cette fragilité.

La Random Forest, elle, teste des seuils sur des pixels individuels et absorbe déjà une partie du
décalage. Elle gagne moins.

---
## 9. Évaluation finale — une seule fois

La validation a tranché : on prend le modèle augmenté. On le réentraîne sur **tout** ce qui n'est
pas le test set, puis on évalue.

Pour être rigoureux, il faudrait augmenter aussi les 10 000 images de validation avant ce
réentraînement final. On le fait.

In [ ]:
t0 = time.perf_counter()

n_val = len(X_val)
X_val_aug = np.empty((n_val * 5, 784), dtype=np.uint8)
y_val_aug = np.empty(n_val * 5, dtype=np.uint8)
X_val_aug[:n_val], y_val_aug[:n_val] = X_val, y_val

for k, (dx, dy) in enumerate(DECALAGES, start=1):
    debut, fin = k * n_val, (k + 1) * n_val
    for i, image in enumerate(X_val):
        X_val_aug[debut + i] = np.clip(shift_image(image, dx, dy), 0, 255)
    y_val_aug[debut:fin] = y_val

X_final = np.vstack([X_train_aug, X_val_aug])
y_final = np.concatenate([y_train_aug, y_val_aug])

perm = rng.permutation(len(X_final))
X_final, y_final = X_final[perm], y_final[perm]

print(f"duree : {time.perf_counter() - t0:.1f} s")
print("training set final :", X_final.shape, f"({X_final.nbytes/1e6:.0f} Mo)")

In [ ]:
rf_final = RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED)

t0 = time.perf_counter()
rf_final.fit(X_final, y_final)
print(f"fit : {time.perf_counter() - t0:.1f} s")

y_pred = rf_final.predict(X_test)
acc_test = accuracy_score(y_test, y_pred)

marge = 1.96 * np.sqrt(acc_test * (1 - acc_test) / len(y_test))
print(f"\nACCURACY SUR LE TEST SET : {acc_test:.4f}")
print(f"IC 95 % : [{acc_test - marge:.4f} ; {acc_test + marge:.4f}]")

---
## 10. Où le gain s'est-il produit ?

Comparons les erreurs des deux modèles sur la validation, classe par classe.

In [ ]:
pred_base = rf_base.predict(X_val)
pred_aug = rf_aug.predict(X_val)

print("Erreurs par classe sur la validation :\n")
print(f"{'classe':>7} {'sans aug':>9} {'avec aug':>9} {'gain':>7}")
for k in range(10):
    m = y_val == k
    e_base = (pred_base[m] != k).sum()
    e_aug = (pred_aug[m] != k).sum()
    print(f"{k:>7} {e_base:>9} {e_aug:>9} {e_base - e_aug:>+7}")

In [ ]:
# Images recuperees par l'augmentation : fausses avant, justes apres
recuperees = (pred_base != y_val) & (pred_aug == y_val)
print(f"{recuperees.sum()} images recuperees par l'augmentation")

imgs = X_val[recuperees]
vrais = y_val[recuperees]
avant = pred_base[recuperees]

n_show = min(10, len(imgs))
fig, axes = plt.subplots(1, n_show, figsize=(1.25 * n_show, 2.1))
for ax, im, v, a in zip(np.atleast_1d(axes), imgs[:n_show], vrais[:n_show], avant[:n_show]):
    ax.imshow(im.reshape(28, 28), cmap="binary")
    ax.set_title(f"{v}\n(etait {a})", fontsize=8)
    ax.axis("off")
fig.suptitle("Images recuperees grace a l'augmentation", y=1.14)
plt.tight_layout()
plt.show()

Beaucoup de ces chiffres sont **mal centrés** ou **écrits de travers**. C'est la signature de ce
qu'on a appris au modèle : la robustesse à la position.

---
## 11. Bilan

### Les quatre pièges de la data augmentation

| Piège | Conséquence | Parade |
|---|---|---|
| Augmenter **avant** le split | Data leakage : les copies d'une image se retrouvent des deux côtés | Split d'abord, augmenter ensuite |
| Augmenter la validation ou le test | On mesure la performance sur des images artificielles | Augmenter le training set seulement |
| Ne pas mélanger | Blocs homogènes, gradients corrélés en SGD | `rng.permutation` après construction |
| Garder du `float64` | 1,6 Go au lieu de 196 Mo | Rester en `uint8` |

### Ce qu'il faut retenir

La data augmentation **n'ajoute pas d'information** : les copies sont déterministes. Elle **injecte
une invariance connue du domaine** que le modèle ne pouvait pas deviner. C'est de la régularisation
au sens du bloc **B13** : on réduit la variance sans toucher au modèle.

Le gain est d'autant plus grand que le modèle était **fragile à la transformation en question**.
Le k-NN, qui compare pixel à pixel, gagne beaucoup. La Random Forest, déjà partiellement robuste,
gagne moins. Un CNN, structurellement invariant par translation, gagnerait encore moins — c'est
tout l'intérêt de son architecture.

### Transposition hors MNIST

La question à se poser pour n'importe quel jeu de données : **quelles transformations laissent
l'étiquette inchangée ?**

- Signal vibratoire d'un banc d'essai : un petit décalage temporel, un léger décalage en fréquence,
  un bruit de mesure réaliste ajouté — la nature du défaut ne change pas.
- Image de contrôle de surface : rotation, symétrie, variation d'éclairage.
- Série de capteurs en vol : fenêtre glissante décalée de quelques échantillons.

La contrainte est toujours la même : la transformation doit être **plausible dans le monde réel**.
Retourner un 6 verticalement donnerait un 9 — l'étiquette changerait, et l'augmentation deviendrait
du bruit étiqueté à tort. C'est pour cela que l'exercice ne retient que les translations.

### Rattachement au carnet

- **B12-3** : réduire la variance sans toucher au bias.
- **B13** : la data augmentation comme régularisation.
- **B13-6** : train / validation / test, et pourquoi le test set reste intact.
- **B1M-3** : data leakage, ici provoqué par un split après augmentation.
- **B11-4** : pourquoi l'ordre compte en stochastic gradient descent.
- **B17-15** : la fragilité du k-NN à la distance, réparée ici.
- **B03-13** : vérifier qu'un gain dépasse la barre d'erreur.